# MergeSortBottomUp (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/MergeSortBottomUp_java.ipynb)

Chapter 18 — Recursive versus iterative algorithms. Runnable notebook of `MergeSortBottomUp`: the algorithm in Java, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/MergeSortBottomUp_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative/MergeSortBottomUp_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `fibRec` (camelCase).

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Recursive MergeSort and MergeSortBottomUp

**MergeSortBottomUp**$(a,n)$ — *Input*: an array $a[1..n]$ of natural numbers. *Output*: the array sorted in non-decreasing order. The same splitting as Merge Sort, but **without recursion**: first we merge pairs of sub-arrays of length $1$, then of length $2$, $4$, … with the procedure `Merge` of chapter 17 (repeated here). It executes $\lceil\log_2 n\rceil$ passes and $\Theta(n\log_2 n)$ steps, like the recursive version, and does not use the call stack.

First the example of the chapter $a=[5,2,4,6,1,3]$, pass by pass (indices are from 0). Then the comparison, on pseudo-random arrays (same generator in all languages, seed $12345$, values between $1$ and $10^6$), of the number of **comparisons** between elements of `InsertionSort`, `MergeSort` and `MergeSortBottomUp`: $\Theta(n^2)$ against $\Theta(n\log_2 n)$, with the two versions of Merge Sort practically equal ("OK" means that the three algorithms give the same sorted array), and the measured times.

In [2]:
%%writefile BottomUp.java
import java.util.*;

public class BottomUp {
    static long comparisons = 0;   // comparisons between elements

    // merge(a, l, m, r) of chapter 17
    static void merge(int[] a, int l, int m, int r) {
        int n1 = m - l + 1;
        int n2 = r - m;
        int[] L = new int[n1];
        int[] R = new int[n2];
        for (int i = 0; i < n1; i++) {
            L[i] = a[l + i];
        }
        for (int j = 0; j < n2; j++) {
            R[j] = a[m + 1 + j];
        }
        int i = 0, j = 0, k = l;
        while (i < n1 && j < n2) {
            comparisons++;
            if (L[i] <= R[j]) {
                a[k] = L[i];
                i++;
            } else {
                a[k] = R[j];
                j++;
            }
            k++;
        }
        while (i < n1) {
            a[k] = L[i];
            i++;
            k++;
        }
        while (j < n2) {
            a[k] = R[j];
            j++;
            k++;
        }
    }

    static void mergeSort(int[] a, int l, int r) {
        if (l < r) {
            int m = l + (r - l) / 2;
            mergeSort(a, l, m);
            mergeSort(a, m + 1, r);
            merge(a, l, m, r);
        }
    }

    // mergeSortBottomUp(a, n): sorts a[0..n-1] without recursion
    static void mergeSortBottomUp(int[] a, int n) {
        int width = 1;
        while (width < n) {
            int l = 0;
            while (l + width < n) {                 // pseudo-code: l + width <= n
                int m = l + width - 1;
                int r = Math.min(l + 2 * width - 1, n - 1);
                merge(a, l, m, r);
                l = l + 2 * width;
            }
            width = 2 * width;
        }
    }

    static void mergeSortBottomUpTrace(int[] a, int n) {
        int width = 1;
        int passes = 0;
        while (width < n) {
            int l = 0;
            StringBuilder parts = new StringBuilder();
            while (l + width < n) {
                int m = l + width - 1;
                int r = Math.min(l + 2 * width - 1, n - 1);
                merge(a, l, m, r);
                if (parts.length() > 0) parts.append(", ");
                parts.append("a[" + l + ".." + m + "] with a[" + (m + 1) + ".." + r + "]");
                l = l + 2 * width;
            }
            passes++;
            System.out.println("width=" + width + ": merge " + parts + " => " + Arrays.toString(a));
            width = 2 * width;
        }
        System.out.println("passes = " + passes);
    }

    // insertionSort(a, n) (chapter 3), with the count of comparisons
    static void insertionSort(int[] a, int n) {
        for (int j = 1; j < n; j++) {
            int key = a[j];
            int i = j - 1;
            while (i >= 0) {
                comparisons++;
                if (a[i] > key) {
                    a[i + 1] = a[i];
                    i = i - 1;
                } else {
                    break;
                }
            }
            a[i + 1] = key;
        }
    }

    // Linear congruential generator: same numbers in Python, C++ and Java
    static int[] randomArray(int n, long seed) {
        long x = seed;
        int[] a = new int[n];
        for (int i = 0; i < n; i++) {
            x = (x * 1103515245L + 12345L) % 2147483648L;
            a[i] = 1 + (int) (x % 1000000L);
        }
        return a;
    }

    static double nowMs() {
        return System.nanoTime() / 1e6;
    }

    public static void main(String[] args) {
        int[] a = {5, 2, 4, 6, 1, 3};
        System.out.println("a = " + Arrays.toString(a) + "   n = " + 6);
        mergeSortBottomUpTrace(a, 6);

        System.out.println();
        System.out.printf("%6s %14s %10s %9s  comparisons%n", "n", "InsertionSort", "MergeSort", "BottomUp");
        double[][] times = new double[3][];
        int idx = 0;
        for (int n : new int[]{1000, 2000, 4000}) {
            int[] base = randomArray(n, 12345);
            int[] x1 = base.clone(), x2 = base.clone(), x3 = base.clone();
            comparisons = 0;
            double t0 = nowMs(); insertionSort(x1, n); double t1 = nowMs() - t0;
            long c1 = comparisons;
            comparisons = 0;
            t0 = nowMs(); mergeSort(x2, 0, n - 1); double t2 = nowMs() - t0;
            long c2 = comparisons;
            comparisons = 0;
            t0 = nowMs(); mergeSortBottomUp(x3, n); double t3 = nowMs() - t0;
            long c3 = comparisons;
            int[] ord = base.clone();
            Arrays.sort(ord);
            String ok = (Arrays.equals(x1, x2) && Arrays.equals(x2, x3) && Arrays.equals(x3, ord)) ? "OK" : "ERROR";
            System.out.printf("%6d %14d %10d %9d  %s%n", n, c1, c2, c3, ok);
            times[idx++] = new double[]{n, t1, t2, t3};
        }
        System.out.println();
        System.out.println("Measured times (they depend on the machine):");
        for (double[] t : times) {
            System.out.printf("n=%-6d InsertionSort %10.3f ms   MergeSort %8.3f ms   MergeSortBottomUp %8.3f ms%n", (int) t[0], t[1], t[2], t[3]);
        }
    }
}

Writing BottomUp.java


In [3]:
!javac BottomUp.java && java BottomUp

a = [5, 2, 4, 6, 1, 3]   n = 6
width=1: merge a[0..0] with a[1..1], a[2..2] with a[3..3], a[4..4] with a[5..5] => [2, 5, 4, 6, 1, 3]
width=2: merge a[0..1] with a[2..3] => [2, 4, 5, 6, 1, 3]
width=4: merge a[0..3] with a[4..5] => [1, 2, 3, 4, 5, 6]
passes = 3

     n  InsertionSort  MergeSort  BottomUp  comparisons
  1000         248782       8716      8740  OK
  2000        1010300      19415     19438  OK
  4000        4027558      42874     42876  OK

Measured times (they depend on the machine):
n=1000   InsertionSort      6.334 ms   MergeSort    1.045 ms   MergeSortBottomUp    0.969 ms
n=2000   InsertionSort      1.677 ms   MergeSort    0.302 ms   MergeSortBottomUp    0.366 ms
n=4000   InsertionSort      3.296 ms   MergeSort    0.577 ms   MergeSortBottomUp    0.980 ms


**Try it yourself.** Change $n$ (with $n=8000$ the comparisons of `InsertionSort` quadruple with respect to $n=4000$, those of Merge Sort little more than double).

In [4]:
%%writefile ProvaBottomUp.java
import java.util.*;

public class ProvaBottomUp {
    public static void main(String[] args) {
        // Try n = 8000: the comparisons of InsertionSort quadruple with respect to n = 4000, those of Merge Sort little more than double.
        int n = 8000;
        int[] base = BottomUp.randomArray(n, 12345);
        int[] x1 = base.clone(), x2 = base.clone();
        BottomUp.comparisons = 0;
        BottomUp.insertionSort(x1, n);
        long c1 = BottomUp.comparisons;
        BottomUp.comparisons = 0;
        BottomUp.mergeSortBottomUp(x2, n);
        System.out.println(c1 + " " + BottomUp.comparisons + " " + (Arrays.equals(x1, x2) ? "True" : "False"));
    }
}

Writing ProvaBottomUp.java


In [5]:
!javac ProvaBottomUp.java && java ProvaBottomUp

16115652 93756 True


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).